In [1]:
# !pip install wandb

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from src.preprocess import BasicDegenderizer, AdvancedDegenderizer
from src.models import DistilBERTClassifier, RoBERTaClassifier

In [4]:
import wandb

In [5]:
wandb.login(key="5489aee4351c1c3af108d0f20e5191f366756c2c")

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hice1/mwesley32/.netrc
wandb: Currently logged in as: mtwesley to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
wandb.init(
    project="NLP-Letters-V2",
    group="nlp-letters-v2-roberta-gendered"
)

In [7]:
DATA_PATH = "data/sentence_sets_trimmed.csv"
LABEL_COLUMN = "applicant_gender"
TEXT_COLUMN = "full_text"
DEGENDERIZERS = ["src/degender/20240301-all.txt"]

In [8]:
# Load dataset
df = pd.read_csv(DATA_PATH, encoding="ISO-8859-1")
print("Dataset shape:", df.shape)

Dataset shape: (3285, 19)


In [9]:
advanced_pipeline = Pipeline([("advanced", AdvancedDegenderizer(paths=DEGENDERIZERS))])
df["degendered"] = advanced_pipeline.fit_transform(df[TEXT_COLUMN].tolist())
df["gendered"] = df[TEXT_COLUMN]

In [10]:
# Convert categorical labels to factors / integers
df[LABEL_COLUMN], class_mapping = pd.factorize(df[LABEL_COLUMN])
print("Class mapping:", dict(enumerate(class_mapping)))

Class mapping: {0: 'male', 1: 'female'}


In [24]:
# Train test splits
X_train, X_test, y_train, y_test = train_test_split(
    df["gendered"],
    df[LABEL_COLUMN],
    test_size=0.2,
    stratify=df[LABEL_COLUMN],
)

print("Train size:", len(X_train), "Test size:", len(X_test))

Train size: 2628 Test size: 657


In [25]:
model = RoBERTaClassifier(
    model_name="roberta-base",
    num_labels=len(class_mapping),
)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [26]:
model.train(
    X_train.tolist(),
    y_train.tolist(),
    epochs=3,
    batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    output_dir="../scratch/nlp-letters-v2-roberta-gendered",    
    report_to=["wandb"]
)

Map:   0%|          | 0/2628 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Auc Roc,Auc Pr,0 Precision,0 Recall,0 F1,1 Precision,1 Recall,1 F1,Cm 00,Cm 01,Cm 10,Cm 11,Runtime,Samples Per Second,Steps Per Second
1,0.001200,0.000458,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,374,0,0,152,5.647800,93.133000,5.843000
2,0.000600,0.000255,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,374,0,0,152,5.649200,93.110000,5.842000
3,0.000500,0.000205,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,374,0,0,152,5.643000,93.213000,5.848000


{'eval_loss': 0.0004578874504659325,
 'eval_accuracy': 1.0,
 'eval_precision': 1.0,
 'eval_recall': 1.0,
 'eval_f1': 1.0,
 'eval_mcc': 1.0,
 'eval_balanced_accuracy': 1.0,
 'eval_cohen_kappa': 1.0,
 'eval_jaccard': 1.0,
 'eval_hamming_loss': 0.0,
 'eval_auc_roc': 1.0,
 'eval_auc_pr': 1.0,
 'eval_0_precision': 1.0,
 'eval_0_recall': 1.0,
 'eval_0_f1': 1.0,
 'eval_1_precision': 1.0,
 'eval_1_recall': 1.0,
 'eval_1_f1': 1.0,
 'eval_cm_00': 374,
 'eval_cm_01': 0,
 'eval_cm_10': 0,
 'eval_cm_11': 152,
 'eval_runtime': 5.6347,
 'eval_samples_per_second': 93.349,
 'eval_steps_per_second': 5.857,
 'epoch': 3.0}

In [27]:
# Inference on de-gendered
model.test(df["degendered"], df[LABEL_COLUMN])

Map:   0%|          | 0/3285 [00:00<?, ? examples/s]

{'accuracy': 0.715372907153729,
 'precision': 0.8576431181485993,
 'recall': 0.500534188034188,
 'f1': 0.41807415934572173,
 'mcc': 0.02764407164110809,
 'balanced_accuracy': 0.500534188034188,
 'cohen_kappa': 0.0015272222986159045,
 'jaccard': 0.3581773061827873,
 'hamming_loss': 0.2846270928462709,
 'auc_roc': 0.500534188034188,
 'auc_pr': 0.285695468914647,
 '0_precision': 0.7152862362971986,
 '0_recall': 1.0,
 '0_f1': 0.8340138469731937,
 '1_precision': 1.0,
 '1_recall': 0.0010683760683760685,
 '1_f1': 0.0021344717182497333,
 'cm_00': 2349,
 'cm_01': 0,
 'cm_10': 935,
 'cm_11': 1}

In [28]:
# Evaluation
model.test(X_test, y_test)

Map:   0%|          | 0/657 [00:00<?, ? examples/s]

{'accuracy': 0.9969558599695586,
 'precision': 0.9978813559322034,
 'recall': 0.9946524064171123,
 'f1': 0.9962502568317239,
 'mcc': 0.9925285100632207,
 'balanced_accuracy': 0.9946524064171123,
 'cohen_kappa': 0.9925005992671818,
 'jaccard': 0.9925337623493157,
 'hamming_loss': 0.0030441400304414,
 'auc_roc': 0.9946524064171123,
 'auc_pr': 0.992348952864666,
 '0_precision': 0.9957627118644068,
 '0_recall': 1.0,
 '0_f1': 0.9978768577494692,
 '1_precision': 1.0,
 '1_recall': 0.9893048128342246,
 '1_f1': 0.9946236559139785,
 'cm_00': 470,
 'cm_01': 0,
 'cm_10': 2,
 'cm_11': 185}